# Ibrahim Time Domain (ITD) Method

The Ibrahim Time Domain (ITD) method is a time-domain modal identification technique used to estimate:

- mode shapes,
- natural frequencies,
- damping ratios

from measured free-decay response data.

Below is a simple implementation procedure.

1. Assume a structure is instrumented with $m$ sensors and its free-decay response is measured over a time duration $T$.  
   The sampled response vector at time $t_k$ is written as

   $$
   \mathbf{x}(t_k)=
   \begin{bmatrix}
   x_1(t_k)\\
   x_2(t_k)\\
   \vdots\\
   x_m(t_k)
   \end{bmatrix},
   \qquad k=1,2,\dots,N
   $$

   where $N$ is the number of sampled time instants.

2. Construct two time-shifted data matrices from the measured response:

   $$
   \mathbf{X}_1 =
   \begin{bmatrix}
   x_1(t_1) & x_1(t_2) & \cdots & x_1(t_{N-1}) \\
   x_2(t_1) & x_2(t_2) & \cdots & x_2(t_{N-1}) \\
   \vdots   & \vdots   & \ddots & \vdots \\
   x_m(t_1) & x_m(t_2) & \cdots & x_m(t_{N-1})
   \end{bmatrix}
   $$

   $$
   \mathbf{X}_2 =
   \begin{bmatrix}
   x_1(t_2) & x_1(t_3) & \cdots & x_1(t_N) \\
   x_2(t_2) & x_2(t_3) & \cdots & x_2(t_N) \\
   \vdots   & \vdots   & \ddots & \vdots \\
   x_m(t_2) & x_m(t_3) & \cdots & x_m(t_N)
   \end{bmatrix}
   $$

3. Estimate the system matrix by

   $$
   \mathbf{A} = \mathbf{X}_2 \mathbf{X}_1^{-1}
   $$

   or, more generally in practice,

   $$
   \mathbf{A} = \mathbf{X}_2 \mathbf{X}_1^{+}
   $$

   where $\mathbf{X}_1^{+}$ is the pseudoinverse of $\mathbf{X}_1$.

   Then compute the eigenvalues and eigenvectors of $\mathbf{A}$:

   $$
   \mathbf{A}\boldsymbol{\phi}_r = \lambda_r \boldsymbol{\phi}_r,
   \qquad r=1,2,\dots,m
   $$

4. The eigenvectors $\boldsymbol{\phi}_r$ represent the mode shapes of the system:

   $$
   \boldsymbol{\phi}_r =
   \begin{bmatrix}
   \phi_{1r}\\
   \phi_{2r}\\
   \vdots\\
   \phi_{mr}
   \end{bmatrix}
   $$

   Each mode shape describes the relative motion of the measured degrees of freedom in mode $r$.

5. The eigenvalues $\lambda_r$ are related to the continuous-time poles $s_r$ by

   $$
   s_r = \frac{1}{\Delta t}\ln(\lambda_r)
   $$

   where $\Delta t$ is the sampling interval. Writing the pole as

   $$
   s_r = \sigma_r + i\omega_{d,r}
   $$

   the damped natural frequency is

   $$
   \omega_{d,r} = \operatorname{Im}(s_r)
   $$

   the undamped natural frequency is

   $$
   \omega_{n,r} = \sqrt{\sigma_r^2 + \omega_{d,r}^2}
   $$

   and the damping ratio is

   $$
   \zeta_r = -\frac{\sigma_r}{\omega_{n,r}}
   $$

The mathematical proof and derivation of the ITD method are beyond the scope of this study.

Also, this is a simple implementation of the method. More advanced and robust formulations are available for complicated systems, noisy measurements, and higher-order modal identification problems.

In [2]:
import numpy as np
from scipy.linalg import eigh

# Parameters
m = 1.0
k = 1.0

# Mass and stiffness matrices
M = np.array([
    [2*m, 0,   0],
    [0,   3*m, 0],
    [0,   0,   1*m]
], dtype=float)

K = np.array([
    [ 5*k, -3*k,  0   ],
    [-3*k,  5*k, -2*k ],
    [ 0,   -2*k,  3*k ]
], dtype=float)

# Solve generalized eigenvalue problem: K phi = lambda M phi
eigvals, eigvecs = eigh(K, M)

# Natural frequencies
omega_n = np.sqrt(eigvals)        # rad/s
f_n = omega_n / (2*np.pi)         # Hz

print("Eigenvalues λ = ω_n^2:")
print(eigvals)
print("\nNatural frequencies ω_n [rad/s]:")
print(omega_n)
print("\nNatural frequencies f_n [Hz]:")
print(f_n)

# For the undamped case
zeta = np.zeros_like(omega_n)
print("\nDamping ratios ζ:")
print(zeta)

Eigenvalues λ = ω_n^2:
[0.42562967 2.741037   4.        ]

Natural frequencies ω_n [rad/s]:
[0.65240299 1.65560774 2.        ]

Natural frequencies f_n [Hz]:
[0.10383316 0.26349816 0.31830989]

Damping ratios ζ:
[0. 0. 0.]


In [1]:
import numpy as np
from scipy.linalg import eig
from scipy.integrate import solve_ivp

# ----------------------------
# 1) Define the 3DOF system
# ----------------------------
m = 1.0
k = 1.0

M = np.array([
    [2*m, 0,   0],
    [0,   3*m, 0],
    [0,   0,   1*m]
], dtype=float)

K = np.array([
    [ 5*k, -3*k,  0   ],
    [-3*k,  5*k, -2*k ],
    [ 0,   -2*k,  3*k ]
], dtype=float)

# Light proportional damping
alpha = 0
beta = 0.
C = alpha * M + beta * K

# ----------------------------
# 2) State-space model
# ----------------------------
Minv = np.linalg.inv(M)

A_top = np.hstack((np.zeros((3, 3)), np.eye(3)))
A_bottom = np.hstack((-Minv @ K, -Minv @ C))
A = np.vstack((A_top, A_bottom))

def state_eq(t, z):
    return A @ z

# Initial condition
x0 = np.array([1.0, 0.5, -0.2])
v0 = np.array([0.0, 0.0, 0.0])
z0 = np.hstack((x0, v0))

dt = 0.01
t_end = 60
t_eval = np.arange(0, t_end, dt)

sol = solve_ivp(state_eq, [0, t_end], z0, t_eval=t_eval, method='RK45')
X = sol.y[:3, :]   # displacement only, shape (3, N)

# ----------------------------
# 3) Build stacked ITD matrices
# ----------------------------
# Stack two consecutive displacement snapshots:
# [x(k); x(k+1)] -> [x(k+1); x(k+2)]
#
# This gives a 6D lifted system, enough for a damped 3DOF case.

N = X.shape[1]

X1 = np.vstack((X[:, 0:N-2], X[:, 1:N-1]))   # shape (6, N-2)
X2 = np.vstack((X[:, 1:N-1], X[:, 2:N]))     # shape (6, N-2)

# Least-squares discrete evolution matrix
A_d = X2 @ np.linalg.pinv(X1)

# Eigenvalues/eigenvectors
mu, V = eig(A_d)

# Continuous-time poles
s = np.log(mu) / dt

# Keep poles with positive imaginary part
selected = np.array([pole for pole in s if np.imag(pole) > 1e-6])

# Sort by damped frequency
selected = selected[np.argsort(np.imag(selected))]

omega_d = np.abs(np.imag(selected))
omega_n = np.sqrt(np.real(selected)**2 + np.imag(selected)**2)
zeta = -np.real(selected) / omega_n
f_n = omega_n / (2 * np.pi)

print("Estimated poles:")
print(selected)

print("\nEstimated damped natural frequencies ω_d [rad/s]:")
print(omega_d)

print("\nEstimated natural frequencies ω_n [rad/s]:")
print(omega_n)

print("\nEstimated natural frequencies f_n [Hz]:")
print(f_n)

print("\nEstimated damping ratios ζ:")
print(zeta)

Estimated poles:
[-1.19206031e-06+0.65240392j -1.82519892e-05+1.65609339j
  3.90733361e-04+2.00159072j]

Estimated damped natural frequencies ω_d [rad/s]:
[0.65240392 1.65609339 2.00159072]

Estimated natural frequencies ω_n [rad/s]:
[0.65240392 1.65609339 2.00159076]

Estimated natural frequencies f_n [Hz]:
[0.10383331 0.26357545 0.31856306]

Estimated damping ratios ζ:
[ 1.82718140e-06  1.10211111e-05 -1.95211414e-04]




## Using Two Consecutive Time Steps

To recover the missing velocity information, two consecutive displacement snapshots are stacked:

$$
\begin{bmatrix}
x(k) \\
x(k+1)
\end{bmatrix}
$$

This stacked vector acts as an **approximate state vector**, because the difference between consecutive displacements implicitly contains velocity information.

The stacked data matrices therefore become

$$
X_1 =
\begin{bmatrix}
x(t_1) & x(t_2) & \cdots & x(t_{N-2}) \\
x(t_2) & x(t_3) & \cdots & x(t_{N-1})
\end{bmatrix}
$$

$$
X_2 =
\begin{bmatrix}
x(t_2) & x(t_3) & \cdots & x(t_{N-1}) \\
x(t_3) & x(t_4) & \cdots & x(t_N)
\end{bmatrix}
$$

Since each displacement vector has dimension $3$, stacking two of them produces

$$
X_1, X_2 \in \mathbb{R}^{6 \times (N-2)}.
$$

The discrete system matrix is then estimated as

$$
A_d = X_2 X_1^{+}
$$

which has dimension

$$
A_d \in \mathbb{R}^{6\times6}.
$$

---

## Result

Because the matrix $A_d$ is now $6 \times 6$, it can produce **six eigenvalues**, corresponding to the **six poles of the damped second-order system**.  

From these poles we obtain the **three natural frequencies**, **three damping ratios**, and the associated **mode shapes**.